In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm

import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


In [2]:
with open('../../transformed_event_logs/artificial_start_end_2_train.pickle', 'rb') as f:
    train_data = pickle.load(f)

In [3]:
train_data

,concept:name,lifecycle:transition_start,time:timestamp_start,org:resource,case:concept:name,id_start,lifecycle:transition_complete,time:timestamp_complete,id_complete,duration,...,Clark,Jane,Joe,Karsten,intercase_n_1__DIAGNOSIS,intercase_n_1__QUALITY_CONTROL,intercase_n_1__REPAIR,intercase_n_3__DIAGNOSIS,intercase_n_3__DIAGNOSIS_REPAIR,intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL
0,DIAGNOSIS,START,2020-01-01 03:57:40.044121+00:00,Jane,0,1,COMPLETE,2020-01-01 04:34:23.549454+00:00,2,0 days 00:36:43.505333,...,0,1,0,0,0,0,0,0,0,0
1,REPAIR,START,2020-01-01 04:34:23.549454+00:00,Joe,0,4,COMPLETE,2020-01-01 14:30:27.423999+00:00,5,0 days 09:56:03.874545,...,0,1,1,0,0,0,0,0,0,0
2,QUALITY_CONTROL,START,2020-01-01 14:30:27.423999+00:00,Joe,0,7,COMPLETE,2020-01-01 22:13:45.345445+00:00,8,0 days 07:43:17.921446,...,0,1,2,0,0,0,0,0,0,0
3,DIAGNOSIS,START,2020-01-01 08:16:35.844753+00:00,Jane,1,10,COMPLETE,2020-01-01 08:47:14.772217+00:00,11,0 days 00:30:38.927464,...,0,1,0,0,0,0,1,0,1,0
4,REPAIR,START,2020-01-01 08:47:14.772217+00:00,Karsten,1,13,COMPLETE,2020-01-01 13:27:22.316694+00:00,14,0 days 04:40:07.544477,...,0,1,0,1,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5399,QUALITY_CONTROL,START,2024-12-26 19:03:41.622915+00:00,Joe,1799,16198,COMPLETE,2024-12-26 21:39:17.312456+00:00,16199,0 days 02:35:35.689541,...,0,0,1,1,1,0,0,1,0,0
5400,DIAGNOSIS,START,2024-12-26 18:05:59.540931+00:00,Clark,1800,16201,COMPLETE,2024-12-26 19:13:26.892809+00:00,16202,0 days 01:07:27.351878,...,1,0,0,0,0,0,1,0,1,0
5401,REPAIR,START,2024-12-26 19:13:26.892809+00:00,Clark,1800,16204,COMPLETE,2024-12-27 08:23:46.946760+00:00,16205,0 days 13:10:20.053951,...,2,0,0,0,0,0,0,0,0,0
5402,QUALITY_CONTROL,START,2024-12-27 08:23:46.946760+00:00,Jane,1800,16207,COMPLETE,2024-12-27 11:10:44.009123+00:00,16208,0 days 02:46:57.062363,...,2,1,0,0,0,0,0,0,0,0


In [4]:
list(train_data.columns)

['concept:name',
 'lifecycle:transition_start',
 'time:timestamp_start',
 'org:resource',
 'case:concept:name',
 'id_start',
 'lifecycle:transition_complete',
 'time:timestamp_complete',
 'id_complete',
 'duration',
 'duration_seconds',
 'seconds_in_day',
 'day_of_week',
 'DIAGNOSIS',
 'QUALITY_CONTROL',
 'REPAIR',
 '1',
 'Clark',
 'Jane',
 'Joe',
 'Karsten',
 'intercase_n_1__DIAGNOSIS',
 'intercase_n_1__QUALITY_CONTROL',
 'intercase_n_1__REPAIR',
 'intercase_n_3__DIAGNOSIS',
 'intercase_n_3__DIAGNOSIS_REPAIR',
 'intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL']

In [5]:
activity_count = [
    'DIAGNOSIS',
    'QUALITY_CONTROL',
    'REPAIR',
 ]

resource_count = [
    '1',
    'Clark',
    'Jane',
    'Joe',
    'Karsten',
]

ii1 = [
    'intercase_n_1__DIAGNOSIS',
    'intercase_n_1__QUALITY_CONTROL',
    'intercase_n_1__REPAIR',
]

ii3 = [
    'intercase_n_3__DIAGNOSIS',
    'intercase_n_3__DIAGNOSIS_REPAIR',
    'intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL'
]

feature_combinations = {
    'A' : ['concept:name'],
    'R' : ['org:resource'],
    'AR' : ['concept:name', 'org:resource'],
    'ARS' : ['concept:name', 'org:resource', 'seconds_in_day'],
    #'ASAC' : ['concept:name', 'seconds_in_day'] + activity_count,
    #'RSRC' : ['org:resource', 'seconds_in_day'] + resource_count,
    'ARSAC' : ['concept:name', 'org:resource', 'seconds_in_day'] + activity_count,
    'ARSRC' : ['concept:name', 'org:resource', 'seconds_in_day'] + resource_count,
    'ARSACRC' : ['concept:name', 'org:resource', 'seconds_in_day'] + activity_count + resource_count,
    #'ARACRC' : ['concept:name', 'org:resource'] + activity_count + resource_count,
    'ARSD' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'],
    'ARSDACRC' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count,
    'ARSDII1' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'] + ii1,
    'ARSDII3' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'] + ii3,
    'ARSDACRCII1' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii1,
    'ARSDACRCII3' : ['concept:name', 'org:resource', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii3
}

In [6]:
quantile_regression_models = {}
for k, v in tqdm(feature_combinations.items()):
    qrm = QuantileRegression(train_data, v)
    qrm.fit()
    quantile_regression_models[k] = qrm

  0%|          | 0/13 [00:00<?, ?it/s]

In [7]:
out_path = './quantile_regression_models.pkl'
with open(out_path, 'wb') as out_file:
    pickle.dump(quantile_regression_models, out_file, protocol=pickle.HIGHEST_PROTOCOL)